# 4) MLP multiclassifier with many hidden layers and comparison of optimization methods

In [1]:
import torch
import torch.nn as nn

from collections.abc import Sequence       # to connect hidden layers sequentially

import torch.optim as optim                # imports PyTorch optimization tools, such as Adam

In [2]:
# IMPORT DATASET MNIST

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from torchvision.datasets import MNIST


data = MNIST('mnist', download=True)

images, labels = data.data, data.targets     #{x_i, y_i}_i : dataset

X = images.view(-1, 28*28).float() / 255     #transforms images into model-ready input


#SPLIT DATASET INTO TRAINING + VALIDATION (unseen data for tuning the model)
X_train, X_valid, y_train, y_valid = train_test_split(X, labels, test_size=0.1, random_state=0)          #10% for validation


In [ ]:
# MAKE MODEL AND DATASET CUDA, to speed up computations !!
# (on this laptop not available)

# 'cuda' already makes the sets as tensors

X_train = X_train.to('cuda')
y_train = y_train.to('cuda')
X_valid = X_valid.to('cuda')
y_valid = y_valid.to('cuda')

model_explicit.cuda()

AssertionError: Torch not compiled with CUDA enabled

>**PyTorch MLP multiclassifier with many hidden layers**:
>
>- class TorchMLP_multiclass(nn.Module): each hidden layer linear + activation function *ReLu*; ouput layer linear with no activation function.
>- cross entropy loss: the *softmax* function for multiclass classification is applied directly by the cross entropy loss, no need to add this information to the class.

Rmk) Networks with more hidden layer have a meaning only if an activation function is applied to each hidden layer.

Rmk) The advantage of *ReLu* is the fact that it is easy to apply and its derivatives are 0 or 1, which speeds the backward phase where the gradient is computed.

Rmk) This is the "network multiclassifier version" of the logistic regression with a single layer.

>**How many hidden layers?**
>
>- low number: MLP learns badly.
>- high number: MLP overfits.
>
>Furthermore, to build deep networks we need to:
>- use GPUs, selecting 'cuda' (instaed than CPUs);
>- normalise the layers, to avoid vanishing or exploding gradient during the optimisation phase and to uniform the scale of weights.

## Scikit-learn, MLP approach

This approach consist in using the 'MLPClassifier' function in Scikit-learn, for multiclass classification. It really slower than employing a PyTorch RealMLP model.


>**Defaults**:
>- 1 hidden layer with 100 units.
>- hidden layers activation function is the ReLu.
>- ouput layer activation function is softmax (for multiclass classification) or sigmoid (for binary classification).

In [3]:
from sklearn.neural_network import MLPClassifier    # library

In [ ]:
classifier_sklearn = MLPClassifier(verbose=1)          # call MLP classifier
                                                       # verbose=1 : prints the loss during the iterations
                                                       
classifier_sklearn.fit(X_train, y_train)      # training

Iteration 1, loss = 0.45203037
Iteration 2, loss = 0.21816061
Iteration 3, loss = 0.16411080
Iteration 4, loss = 0.13230531
Iteration 5, loss = 0.10896826
Iteration 6, loss = 0.09206867
Iteration 7, loss = 0.08011522
Iteration 8, loss = 0.06869145
Iteration 9, loss = 0.06044620
Iteration 10, loss = 0.05377318
Iteration 11, loss = 0.04729932
Iteration 12, loss = 0.04239229
Iteration 13, loss = 0.03761330
Iteration 14, loss = 0.03410461
Iteration 15, loss = 0.03001103
Iteration 16, loss = 0.02738373
Iteration 17, loss = 0.02437567
Iteration 18, loss = 0.02137731
Iteration 19, loss = 0.01952485
Iteration 20, loss = 0.01791866
Iteration 21, loss = 0.01575867
Iteration 22, loss = 0.01490339
Iteration 23, loss = 0.01267452
Iteration 24, loss = 0.01104196
Iteration 25, loss = 0.00993519
Iteration 26, loss = 0.00951989
Iteration 27, loss = 0.00854452
Iteration 28, loss = 0.00764924
Iteration 29, loss = 0.00689785
Iteration 30, loss = 0.00594063
Iteration 31, loss = 0.00569541
Iteration 32, los

,"verbose verbose: bool, default=FalseWhether to print progress messages to stdout.",1
,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(100,)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


In [8]:
prediction = classifier_sklearn.predict(X_valid)      # prediction

accuracy_score(y_valid, prediction)           # evaluation

0.9783333333333334

We can increase the number of units in the hidden layer or the number of hidden layers, but computations will be extremely slow.

In [ ]:
# 800 units in 1 hidden layer

prediction2 = MLPClassifier(hidden_layer_sizes=(800,)).fit(X_train, y_train).predict(X_valid)
accuracy_score(y_valid, prediction2)

In [ ]:
# 100, 200, 100 units in 3 hidden layers respectively

prediction3 = MLPClassifier(hidden_layer_sizes=(100,200,100)).fit(X_train, y_train).predict(X_valid)
accuracy_score(y_valid, prediction3)

## MLP multiclass classifier, explicit version with 3 hidden layers

*Forward phase*: ReLu is applied to all hidden layers, but not to the output layer.

In [3]:
class TorchMLP_explicit(nn.Module):

  # define: 3 LINEAR LAYERS CONNECTIONS + 1 ACTIVATION FUNCTION ReLu
  def __init__(self, input_dim, hidden_dim_1, hidden_dim_2, hidden_dim_3, output_dim):

    super().__init__()

    self.linear1 = nn.Linear(input_dim, hidden_dim_1)
    self.linear2 = nn.Linear(hidden_dim_1, hidden_dim_2)
    self.linear3 = nn.Linear(hidden_dim_2, hidden_dim_3)
    self.linear4 = nn.Linear(hidden_dim_3, output_dim)

    self.relu = nn.ReLU()

  # define FORWARD PHASE: linear functions + activation function within hidden layers 
  def forward(self, x):

    x = self.relu(self.linear1(x))      # applies ReLu between hidden layers,
    x = self.relu(self.linear2(x))
    x = self.relu(self.linear3(x))

    return self.linear4(x)              # but not to output layer



In [4]:
# call model

model_explicit = TorchMLP_explicit(784, 100, 200, 100, 10)


In [5]:
# PARAMETERS BETWEEN LAYERS

# .parameters() contains the number of weights + biases between each layer 

for param in model_explicit.parameters():
  print(param.numel())

# TOTAL NUMBER OF WEIGHTS

print('Total : ', sum(param.numel() for param in model_explicit.parameters() ))

78400
100
20000
200
20000
100
1000
10
Total :  119810


In [6]:
loss_explicit = nn.CrossEntropyLoss()                          # cross entropy loss for multiclass classification
optimizer_explicit = optim.Adam(model_explicit.parameters())     # optimizer Adam for weights update

In [7]:
# TRAINING AND EVALUATION for 1000 times

for epoch in range(1000):     

  #TRAINING

  optimizer_explicit.zero_grad()                      # forget gradient
  y_pred = model_explicit(X_train)                    # predict logits
  loss_result = loss_explicit(y_pred, y_train)        # apply softmax and compute loss   
  loss_result.backward()                              # compute gradient backwards                 
  optimizer_explicit.step()                           # optimize weights               


  # EVALUATION every 50 epochs

  if epoch % 50 == 0:                              
    with torch.no_grad():                               # clean gradient, to save memory
      pred_valid = model_explicit(X_valid)

    predictions = torch.argmax(pred_valid, dim=1)       # output classes 
    print(accuracy_score(y_valid, predictions.to('cpu')))


0.09866666666666667
0.9076666666666666
0.9425
0.9583333333333334
0.9666666666666667
0.9683333333333334
0.9693333333333334
0.9696666666666667
0.9698333333333333
0.97
0.97
0.9701666666666666
0.9718333333333333
0.9723333333333334
0.972
0.972
0.9721666666666666
0.9725
0.9726666666666667
0.973


## MLP multiclass classification, implicit version

Fast method to define hidden layers.

In [8]:

class TorchMLP(nn.Module):
    
    def __init__(self, input_dim: int, hidden_dims: Sequence[int], output_dim: int):
        
        super().__init__()

        dims = [input_dim, *hidden_dims, output_dim]                   # list of layers dimensions / number of units

        layers = []                                                    # contains fully connections of layers / arrows and activation functions

        for i, (in_dim, out_dim) in enumerate(zip(dims, dims[1:])):    # enumerate: run on couples
                                                                       # zip : pairs each couple
                                                                       # [1: ] : takes off element 0
                                                                       # connect adjacent layers
            
            layers.append(nn.Linear(in_dim, out_dim))                  # linearly

            layers.append(nn.ReLU())                                   # add ReLu after each hidden layer


        layers = layers[:-1]                                           # remove ReLu applied to output

        self.model = nn.Sequential(*layers)                            # run layers sequentially

    def forward(self, x):                                              # forward computations sequential
        
        return self.model(x)


In [9]:
#CALL THE MODEL

hidden_layers_dim = [100, 200, 100]

model = TorchMLP(784, hidden_layers_dim, 10)

# LOSS

loss = nn.CrossEntropyLoss()


In [10]:
# PARAMETERS BETWEEN LAYERS

# .parameters() contains the number of weights + biases between each layer 

for param in model.parameters():
  print(param.numel())

# TOTAL NUMBER OF WEIGHTS

print('Total : ', sum(param.numel() for param in model.parameters() ))

78400
100
20000
200
20000
100
1000
10
Total :  119810


## Comparison of optimization methods (continuation of implicit method)

We consider the implicit MLP for multiclass classification.

**OPTIMIZATION PHASE:** an optimization algorithm minimizes the loss function $L$ over the weights and updates the weights as:
$$
W_{i} \leftarrow W_{i} - \nu \cdot \nabla_{W_{i}}(L),
$$
where $\nu$ is the learning rate.

The gradient is subtracted from the weights with a step of $\nu$ at each iteration of the algorithm.

**RESULTS**: among SDG and Adam optimization methods, Adam provides best results in terms of accuracy.

### Gradient Descent

> Classic version, where $\nu > 0$ is constant (and small).

### Mini-batch Stochastic Gradient Descent (Mini-batch SDG)

> - The training dataset is divided into small random subsets of the same size called mini-batches (by Dataloader).
> - The training pahse is performed on each mini-batch, one at a time, until all of them are visited.

This method increases the randomness, and avoid that the loss gets stucked into false minima or flat regions.

> **SDG**: mini-batch of size 1, coinciding with the single samples.

**This method comes naturally when using Dataloader + SGD as optimization method**.

*The mini-batch training can be eventually combined with other optimizer, such as Adam.*

### Stochastic gradient descent (SGD)

> Stochastic version of the gradient descent, where the optimization phase is performed on one sample at a time, until all samples have been visited.

> SGD == Mini-Batch SGD for batch_size=1

In [11]:
# OPTIMIZER SDG, with learning rate lr

optimizer_sdg = torch.optim.SGD(model.parameters(), lr=0.01)   
  

In [13]:
# TRAINING AND EVALUATION for 1000 times

for epoch in range(1000):     

  #TRAINING

  optimizer_sdg.zero_grad()
  y_pred = model(X_train)              
  loss_result = loss(y_pred, y_train)       
  loss_result.backward()                          
  optimizer_sdg.step()                             


  # EVALUATION

  if epoch % 50 == 0:                              
    with torch.no_grad():                        # clean gradient, to save memory
      pred_valid = model(X_valid)

    predictions = torch.argmax(pred_valid, dim=1)
    print(accuracy_score(y_valid, predictions))


0.09733333333333333
0.09733333333333333
0.09733333333333333
0.0975
0.10216666666666667
0.12183333333333334
0.197
0.3285
0.39916666666666667
0.426
0.42766666666666664
0.4195
0.4098333333333333
0.4043333333333333
0.4186666666666667
0.4495
0.4876666666666667
0.5335
0.5731666666666667
0.6138333333333333


#### SDG by hand function:

In [117]:
class MySGD():

    def __init__(self, params, lr):
        self.params = list(params)                      # store parameters as a list
        self.lr = lr

    def step(self):                                     # weights update
        for param in self.params:
            param.data -= self.lr * param.grad

    def zero_grad(self):                                # sets gradients to None
        for param in self.params:
            param.grad = None
  

### Adam

> Adaptative method, where the learning rate is not constant and adapts to the gradient size at every iteration. If the gradient is large, the learning rate decreases. If the gradient is small, the learning rate increases. Thanks to this, Adam avoids that during the minimization process, the loss gets stucked in false minima or flat regions.

In [ ]:
#OPTIMIZER Adam

optimizer_adam = optim.Adam(model.parameters(), lr=0.002)   # lr = 0.001 by default
                                                            # lr is only used to fix the scale of the learning rate


In [ ]:
# TRAINING AND EVALUATION for 1000 times

for epoch in range(1000):     

  #TRAINING

  optimizer_adam.zero_grad()
  y_pred = model(X_train)              
  loss_result = loss(y_pred, y_train)       
  loss_result.backward()                          
  optimizer_adam.step()                             


  # EVALUATION

  if epoch % 50 == 0:                              
    with torch.no_grad():                               # clean gradient, to save memory
      pred_valid = model(X_valid)

    _, predictions = torch.argmax(pred_valid, dim=1)
    print(accuracy_score(y_valid, predictions))


C:\Users\valen\AppData\Local\Temp\ipykernel_14456\4251778128.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loss_result = loss(y_pred, torch.tensor(y_train))
C:\Users\valen\AppData\Local\Temp\ipykernel_14456\4251778128.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pred_valid = model(torch.tensor(X_valid))
C:\Users\valen\AppData\Local\Temp\ipykernel_14456\4251778128.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loss_result = loss(y_pred, torch.tensor(y_train))


0.5826666666666667


C:\Users\valen\AppData\Local\Temp\ipykernel_14456\4251778128.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pred_valid = model(torch.tensor(X_valid))
C:\Users\valen\AppData\Local\Temp\ipykernel_14456\4251778128.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loss_result = loss(y_pred, torch.tensor(y_train))


0.9228333333333333


C:\Users\valen\AppData\Local\Temp\ipykernel_14456\4251778128.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pred_valid = model(torch.tensor(X_valid))


0.956


... (30 outputs left out)

C:\Users\valen\AppData\Local\Temp\ipykernel_14456\4251778128.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pred_valid = model(torch.tensor(X_valid))
C:\Users\valen\AppData\Local\Temp\ipykernel_14456\4251778128.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loss_result = loss(y_pred, torch.tensor(y_train))


0.972


C:\Users\valen\AppData\Local\Temp\ipykernel_14456\4251778128.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pred_valid = model(torch.tensor(X_valid))
C:\Users\valen\AppData\Local\Temp\ipykernel_14456\4251778128.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loss_result = loss(y_pred, torch.tensor(y_train))


0.9716666666666667
